# Dia 7 (extensao) - Modelo de risco de queda atipica para 2024

**Objetivo:** treinar um classificador que aprende, a partir do perfil de um
municipio ate o ano T, a probabilidade dele ter uma queda atipica de
cobertura vacinal no ano T+1 (rotulo historico ja existente em
`data/processed/destaques_quedas_atipicas.csv`, Dia 5 Parte 2). Depois,
aplicar esse modelo ao snapshot mais recente (2014-2023 inteiro) pra gerar
um "risco para 2024" por municipio.

**O problema central -- vazamento temporal:** as features que ja existem em
`data/processed/clusters_municipios_mvp.csv` (`media_cobertura`,
`variabilidade_cobertura`, `tendencia_cobertura`, `isolamento_similaridade`,
`cluster`) foram calculadas sobre a serie INTEIRA 2014-2023. Usa-las direto
pra prever o ano T+1 de qualquer T < 2023 vaza informacao do futuro pro
passado -- ex: a `tendencia_cobertura` de um municipio "calculada hoje" ja
sabe o que aconteceu entre T+1 e 2023, coisa que nao estaria disponivel no
momento real da decisao. Pra treinar sem vazamento, recalculamos todas as
features de forma **causal/rolling**: pra cada (municipio, ano T) que vira
uma linha de treino, as features usam SO dados ate o ano T.

Isso vale inclusive pras duas features que comparam municipios entre si
(`isolamento_similaridade`, grafo k-NN, e `cluster`, KMeans) -- que nao sao
series por municipio, e sim derivadas de comparar TODOS os municipios num
unico corte no tempo. Pra ficarem causais, o pipeline do Dia 4/5 (padronizar
-> KMeans K=3 -> k-NN K=5 -> isolamento) e repetido pra CADA ano T, usando
so as features causais calculadas até aquele T -- nao reaproveita o
resultado unico de 2014-2023.

Decisoes de judgment deste notebook (confirmadas com o usuario antes de
implementar): recalcular causalmente isolamento/cluster (nao excluir essas
features nem congela-las); lidar com o desbalanceamento (~3% positivos) via
peso de classe + AUC-ROC/AUC-PR (nao undersampling); regressao logistica em
numpy puro, mesmo padrao do KMeans do Dia 4 (bloqueio de scipy/Device
Guard nesta maquina); treino OFFLINE, artefato salvo em
`data/processed/modelo_risco.json`, backend so carrega e aplica
`predict()` por request.

In [1]:
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_columns", None)

## Etapa A - features causais por (municipio, ano T)

**Conceito:** pra cada municipio, pra cada ano T de 2014 a 2023, calculamos
`media_causal(T)`, `variabilidade_causal(T)` e `tendencia_causal(T)` usando
SO os anos <= T da serie "cobertura geral" (media das 5 vacinas, mesma
logica do Dia 4 Parte 1). Isso e diferente de um rolling window de tamanho
fixo -- aqui a janela cresce (expanding), porque queremos "tudo que esse
municipio tinha de historico disponivel naquele momento", nao um recorte
fixo de N anos.

**Regra de elegibilidade (nova, analoga a regra >=7/10 anos do Dia 4):**
so entra na tabela causal uma linha (municipio, T) com pelo menos 3 anos
validos de historico ATE T -- com 1 ou 2 pontos, desvio padrao e slope de
regressao ficam instaveis/sem sentido estatistico. Isso elimina
automaticamente T=2014 e boa parte de T=2015 da base de treino (nao da pra
calcular tendencia confiavel com 1-2 anos), mas nao afeta os anos usados de
fato no modelo (T=2016 em diante).

In [2]:
# Passo 1: cobertura geral por (ibge6, ano) -- mesma logica do Dia 4/5
serie = pd.read_csv(
    "../data/processed/serie_temporal_mvp.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
)
cobertura_geral = serie.groupby(["ibge6", "ano"])["cobertura_capada"].mean().reset_index()
cobertura_geral_valid = (
    cobertura_geral.dropna(subset=["cobertura_capada"]).sort_values(["ibge6", "ano"])
)

print(f"Linhas da serie temporal: {len(serie)}")
print(f"(ibge6, ano) com cobertura geral valida: {len(cobertura_geral_valid)}")

Linhas da serie temporal: 278500
(ibge6, ano) com cobertura geral valida: 55684


In [3]:
# Passo 2: features causais (expanding) por municipio.
#
# Pra cada municipio, percorre os anos validos em ordem e, em cada posicao j,
# calcula media/desvio/slope usando SO os pontos de 0 a j (inclusive) --
# ou seja, "o que esse municipio sabia sobre si mesmo" naquele ano.

def causal_features_grupo(grupo):
    grupo = grupo.sort_values("ano")
    anos = grupo["ano"].values.astype(float)
    valores = grupo["cobertura_capada"].values
    linhas = []
    for j in range(len(valores)):
        anos_j = anos[: j + 1]
        valores_j = valores[: j + 1]
        n_validos = j + 1
        media = valores_j.mean()
        if n_validos >= 2:
            variab = valores_j.std(ddof=1)
            slope = np.polyfit(anos_j, valores_j, 1)[0]
        else:
            variab = np.nan
            slope = np.nan
        linhas.append({
            "ano": anos[j],
            "n_validos_causal": n_validos,
            "media_causal": media,
            "variabilidade_causal": variab,
            "tendencia_causal": slope,
        })
    return pd.DataFrame(linhas)


causal = (
    cobertura_geral_valid.groupby("ibge6")
    .apply(causal_features_grupo, include_groups=False)
    .reset_index()
    .drop(columns=["level_1"])
)
causal["ano"] = causal["ano"].astype(int)

print(f"Tabela causal: {causal.shape}")
print("\nExemplo (Alta Floresta D'Oeste, RO, ibge6=110001) -- note que media/variabilidade/")
print("tendencia mudam a cada T conforme mais historico vai ficando disponivel:")
causal[causal["ibge6"] == "110001"]

Tabela causal: (55684, 6)

Exemplo (Alta Floresta D'Oeste, RO, ibge6=110001) -- note que media/variabilidade/
tendencia mudam a cada T conforme mais historico vai ficando disponivel:


,ibge6,ano,n_validos_causal,media_causal,variabilidade_causal,tendencia_causal
0,110001,2014,1,92.785940,NaN,NaN
1,110001,2015,2,85.964020,9.647652,-13.643840
2,110001,2016,3,83.843500,7.747800,-6.591740
3,110001,2017,4,85.100140,6.807020,-1.128728
4,110001,2018,5,86.329200,6.504197,0.664696
5,110001,2019,6,87.245267,6.235281,1.165026
6,110001,2020,7,87.883331,5.937072,1.206690
7,110001,2021,8,86.736847,6.381903,0.040137
8,110001,2022,9,85.444831,7.117679,-0.747114
9,110001,2023,10,85.479984,6.711533,-0.524181


## Etapa B - isolamento e cluster causais por ano T

**Conceito:** o pipeline do Dia 4 (padronizar -> KMeans K=3) e do Dia 5
(k-NN K=5 -> distancia media = isolamento) e repetido pra CADA ano T,
usando so os municipios que ja tem >= 3 anos causais ate T e so as 3
features causais DAQUELE T. Isso significa rodar KMeans e k-NN **9 vezes**
(uma por T de 2015 a 2023) -- mais trabalho que uma serie simples, mas e o
preco de nao vazar informacao futura num sinal que, por natureza, compara
todos os municipios entre si.

As funcoes de KMeans/k-NN sao as mesmas do Dia 4 (notebook02) e Dia 5
(notebook03), em numpy puro -- so reaproveitadas aqui, nao reescritas.

Validacao esperada: em T=2023 (ultimo ano), "causal" = usar o historico
inteiro 2014-2023 -- ou seja, o resultado DEVE bater exatamente com o
`clusters_municipios_mvp.csv` ja existente (3 clusters de tamanho 2.860 /
1.391 / 1.319). Isso confirma que a logica causal esta implementada
corretamente antes de confiar nela pros anos intermediarios.

In [4]:
# KMeans e k-NN em numpy puro -- mesmas funcoes do Dia 4 (notebook02) e
# Dia 5 (notebook03), reaproveitadas sem alteracao de logica.

def kmeans_plus_plus_init(X, k, rng):
    n = X.shape[0]
    centros = np.empty((k, X.shape[1]))
    centros[0] = X[rng.integers(n)]
    dist_sq = np.sum((X - centros[0]) ** 2, axis=1)
    for i in range(1, k):
        probs = dist_sq / dist_sq.sum()
        centros[i] = X[rng.choice(n, p=probs)]
        nova_dist_sq = np.sum((X - centros[i]) ** 2, axis=1)
        dist_sq = np.minimum(dist_sq, nova_dist_sq)
    return centros


def kmeans_lloyd(X, k, centros_iniciais, max_iter=300, tol=1e-4):
    centros = centros_iniciais.copy()
    for _ in range(max_iter):
        dists = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
        labels = dists.argmin(axis=1)
        novos_centros = centros.copy()
        for j in range(k):
            membros = X[labels == j]
            if len(membros) > 0:
                novos_centros[j] = membros.mean(axis=0)
        deslocamento = np.sum((novos_centros - centros) ** 2)
        centros = novos_centros
        if deslocamento < tol:
            break
    dists_finais = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    labels = dists_finais.argmin(axis=1)
    inertia = dists_finais[np.arange(len(X)), labels].sum()
    return labels, centros, inertia


def kmeans_proprio(X, k, n_init=10, random_state=42, max_iter=300):
    rng_principal = np.random.default_rng(random_state)
    melhor_inertia = np.inf
    melhor_labels = melhor_centros = None
    for _ in range(n_init):
        rng = np.random.default_rng(rng_principal.integers(0, 2**32 - 1))
        centros_iniciais = kmeans_plus_plus_init(X, k, rng)
        labels, centros, inertia = kmeans_lloyd(X, k, centros_iniciais, max_iter=max_iter)
        if inertia < melhor_inertia:
            melhor_inertia, melhor_labels, melhor_centros = inertia, labels, centros
    return melhor_labels, melhor_centros, melhor_inertia


def knn_distancias(X, k=5, batch=500):
    n = X.shape[0]
    vizinhos_dist = np.zeros((n, k))
    for inicio in range(0, n, batch):
        fim = min(inicio + batch, n)
        bloco = X[inicio:fim]
        diffs = bloco[:, None, :] - X[None, :, :]
        dists = np.sqrt((diffs ** 2).sum(axis=2))
        for i_local, i_global in enumerate(range(inicio, fim)):
            dists[i_local, i_global] = np.inf
        idx = np.argsort(dists, axis=1)[:, :k]
        for i_local, i_global in enumerate(range(inicio, fim)):
            vizinhos_dist[i_global] = dists[i_local, idx[i_local]]
    return vizinhos_dist

In [5]:
# Roda o pipeline causal pra cada T de 2014 a 2023 (T=2014/2015 ficam vazios
# pela regra de >=3 anos). Nomeia os clusters pela mesma heuristica do Dia 5
# (maior media = "alta_estavel"; entre os 2 restantes, maior tendencia =
# "baixa_recuperacao", menor tendencia = "queda_instavel") pra manter
# categorias comparaveis entre anos diferentes, apesar do indice numerico
# do KMeans ser arbitrario a cada rodada.

MIN_ANOS_CAUSAL = 3
COLUNAS_3F = ["media_causal", "variabilidade_causal", "tendencia_causal"]

resultados_por_T = {}
for T in range(2014, 2024):
    sub = causal[(causal["ano"] == T) & (causal["n_validos_causal"] >= MIN_ANOS_CAUSAL)].copy()
    if len(sub) < 10:
        continue

    X = sub[COLUNAS_3F].values
    media_c, desvio_c = X.mean(axis=0), X.std(axis=0, ddof=0)
    Xp = (X - media_c) / desvio_c

    labels, _, _ = kmeans_proprio(Xp, k=3, n_init=10, random_state=42)
    sub["cluster_raw"] = labels

    stats = sub.groupby("cluster_raw")[COLUNAS_3F].mean()
    alta = stats["media_causal"].idxmax()
    demais = stats.index.difference([alta])
    recuperacao = stats.loc[demais, "tendencia_causal"].idxmax()
    queda = stats.loc[demais, "tendencia_causal"].idxmin()
    sub["cluster_causal"] = sub["cluster_raw"].map(
        {alta: "alta_estavel", recuperacao: "baixa_recuperacao", queda: "queda_instavel"}
    )

    dist5 = knn_distancias(Xp, k=5, batch=500)
    sub["isolamento_causal"] = dist5.mean(axis=1)

    resultados_por_T[T] = sub[
        ["ibge6", "ano", "media_causal", "variabilidade_causal", "tendencia_causal",
         "cluster_causal", "isolamento_causal"]
    ]
    print(f"T={T}: n={len(sub)} municipios, clusters={sub['cluster_causal'].value_counts().to_dict()}")

causal_completo = pd.concat(resultados_por_T.values(), ignore_index=True)
print(f"\nTabela causal completa: {causal_completo.shape}")

T=2016: n=5561 municipios, clusters={'alta_estavel': 3305, 'baixa_recuperacao': 1589, 'queda_instavel': 667}


T=2017: n=5569 municipios, clusters={'alta_estavel': 3243, 'baixa_recuperacao': 1424, 'queda_instavel': 902}


T=2018: n=5570 municipios, clusters={'alta_estavel': 3257, 'baixa_recuperacao': 1202, 'queda_instavel': 1111}


T=2019: n=5568 municipios, clusters={'alta_estavel': 3376, 'queda_instavel': 1382, 'baixa_recuperacao': 810}


T=2020: n=5568 municipios, clusters={'alta_estavel': 2929, 'baixa_recuperacao': 1339, 'queda_instavel': 1300}


T=2021: n=5569 municipios, clusters={'alta_estavel': 2816, 'queda_instavel': 1377, 'baixa_recuperacao': 1376}


T=2022: n=5569 municipios, clusters={'alta_estavel': 2859, 'baixa_recuperacao': 1357, 'queda_instavel': 1353}


T=2023: n=5570 municipios, clusters={'alta_estavel': 2860, 'baixa_recuperacao': 1391, 'queda_instavel': 1319}

Tabela causal completa: (44544, 7)


In [6]:
# Validacao: em T=2023 (historico inteiro = o mesmo usado no Dia 4/5), os
# tamanhos de cluster DEVEM bater com clusters_municipios_mvp.csv (2.860 /
# 1.391 / 1.319) -- confirma que a logica causal nao introduziu nenhuma
# diferenca espuria em relacao ao pipeline original quando T = ultimo ano.

clusters_originais = pd.read_csv(
    "../data/processed/clusters_municipios_mvp.csv", dtype={"ibge6": str}
)
print("Tamanhos originais (Dia 4, 2014-2023 inteiro):")
print(clusters_originais["nome_cluster"].value_counts())

print("\nTamanhos causais em T=2023 (deve bater):")
print(causal_completo[causal_completo["ano"] == 2023]["cluster_causal"].value_counts())

causal_completo.to_csv("../data/processed/features_causais_risco.csv", index=False)
print("\nSalvo em data/processed/features_causais_risco.csv (intermediario, auditoria)")

Tamanhos originais (Dia 4, 2014-2023 inteiro):
nome_cluster
Cobertura alta e estavel           2860
Cobertura baixa, em recuperacao    1391
Cobertura em queda e instavel      1319
Name: count, dtype: int64

Tamanhos causais em T=2023 (deve bater):
cluster_causal
alta_estavel         2860
baixa_recuperacao    1391
queda_instavel       1319
Name: count, dtype: int64



Salvo em data/processed/features_causais_risco.csv (intermediario, auditoria)


## Etapa C - tabela de treino: rotulo e split temporal

**Rotulo:** para cada linha (municipio, T), `y=1` se esse municipio aparece
em `destaques_quedas_atipicas.csv` no ano **T+1** (queda atipica, z < -2,
Dia 5 Parte 2), senao `y=0`. So entram linhas com T <= 2022 (T+1 precisa
existir nos dados, ate 2023).

**Split temporal (nao aleatorio):** treino nos anos T=2016..2020 (preve
2017-2021), teste em T=2021..2022 (preve 2022-2023 -- os 2 anos mais
recentes, os mais parecidos com a aplicacao real do modelo em 2023->2024).
Um split aleatorio linha-a-linha vazaria informacao: o MESMO municipio
aparece em varios anos, e anos vizinhos de um mesmo municipio sao
correlacionados (ele nao muda de perfil do nada de um ano pro outro) --
treinar num ano e testar no ano vizinho do mesmo municipio superestimaria o
desempenho do modelo.

In [7]:
# Monta a tabela de treino: features causais de T + rotulo (queda atipica em T+1)
destaques = pd.read_csv(
    "../data/processed/destaques_quedas_atipicas.csv", dtype={"ibge6": str}
)
quedas_por_ibge6_ano = set(zip(destaques["ibge6"], destaques["ano"]))

df = causal_completo[causal_completo["ano"] <= 2022].copy()
df["ano_t1"] = df["ano"] + 1
df["y"] = [
    1 if (ibge6, ano_t1) in quedas_por_ibge6_ano else 0
    for ibge6, ano_t1 in zip(df["ibge6"], df["ano_t1"])
]

print(f"Linhas de treino totais (T=2016..2022): {len(df)}")
print(f"Positivos: {df['y'].sum()} ({100 * df['y'].mean():.2f}%)")
print("\nPositivos por ano T (preve T+1):")
print(df.groupby("ano")["y"].agg(n_positivos="sum", n_total="count", taxa="mean"))

Linhas de treino totais (T=2016..2022): 38974
Positivos: 1076 (2.76%)

Positivos por ano T (preve T+1):
      n_positivos  n_total      taxa
ano                                 
2016          100     5561  0.017982
2017          172     5569  0.030885
2018          165     5570  0.029623
2019          194     5568  0.034842
2020          180     5568  0.032328
2021          112     5569  0.020111
2022          153     5569  0.027474


In [8]:
# one-hot do cluster causal (baseline = alta_estavel, fica implicito quando
# as duas dummies abaixo sao 0)
df["cluster_baixa_recuperacao"] = (df["cluster_causal"] == "baixa_recuperacao").astype(float)
df["cluster_queda_instavel"] = (df["cluster_causal"] == "queda_instavel").astype(float)

FEATURES_CONTINUAS = ["media_causal", "variabilidade_causal", "tendencia_causal", "isolamento_causal"]
FEATURES_DUMMIES = ["cluster_baixa_recuperacao", "cluster_queda_instavel"]
FEATURES = FEATURES_CONTINUAS + FEATURES_DUMMIES

TREINO_T = list(range(2016, 2021))   # preve 2017-2021
TESTE_T = [2021, 2022]               # preve 2022-2023

treino = df[df["ano"].isin(TREINO_T)].copy()
teste = df[df["ano"].isin(TESTE_T)].copy()

print(f"Treino: {len(treino)} linhas, {int(treino['y'].sum())} positivos ({100*treino['y'].mean():.2f}%)")
print(f"Teste:  {len(teste)} linhas, {int(teste['y'].sum())} positivos ({100*teste['y'].mean():.2f}%)")

Treino: 27836 linhas, 811 positivos (2.91%)
Teste:  11138 linhas, 265 positivos (2.38%)


## Etapa D - regressao logistica em numpy, com peso de classe

**Conceito regressao logistica:** modelo linear `z = intercepto + soma(peso_i * feature_i)`,
passado por uma sigmoide `p = 1/(1+exp(-z))` pra virar uma probabilidade
entre 0 e 1. Treina por gradiente descendente, minimizando a entropia
cruzada (cross-entropy) entre `p` e o rotulo real `y`.

**Desbalanceamento (~3% positivos), decisao ja confirmada:** peso de
classe, formula "balanced" (igual ao `class_weight="balanced"` do sklearn):
`peso = n_total / (2 * n_da_classe)`. Isso faz cada erro na classe rara
(queda atipica) pesar ~17x mais que um erro na classe comum no gradiente --
sem isso, o modelo aprenderia a simplesmente prever "nunca" e já acertaria
~97% (acuracia alta, mas inutil). Por isso avaliamos com AUC-ROC e AUC-PR
(que nao sao inflados por essa assimetria), nao acuracia.

Padronizacao das 4 features continuas e feita com media/desvio do TREINO
apenas (nunca do teste, pra nao vazar a distribuicao dos dados de teste pro
treino).

In [9]:
# Padronizacao (fit so no treino)
media_treino = treino[FEATURES_CONTINUAS].mean().values
desvio_treino = treino[FEATURES_CONTINUAS].std(ddof=0).values

def monta_X(sub):
    cont_padr = (sub[FEATURES_CONTINUAS].values - media_treino) / desvio_treino
    return np.hstack([cont_padr, sub[FEATURES_DUMMIES].values])

X_treino, y_treino = monta_X(treino), treino["y"].values.astype(float)
X_teste, y_teste = monta_X(teste), teste["y"].values.astype(float)

# pesos de classe (balanced)
n = len(y_treino)
n_pos, n_neg = y_treino.sum(), n - y_treino.sum()
peso_pos, peso_neg = n / (2 * n_pos), n / (2 * n_neg)
pesos_treino = np.where(y_treino == 1, peso_pos, peso_neg)

print(f"peso classe positiva (queda atipica): {peso_pos:.2f}")
print(f"peso classe negativa: {peso_neg:.2f}")

peso classe positiva (queda atipica): 17.16
peso classe negativa: 0.52


In [10]:
def treinar_logistica(X, y, pesos, lr=0.3, n_iter=8000, l2=1e-3):
    n, d = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])  # coluna de 1s para o intercepto
    beta = np.zeros(d + 1)
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-(Xb @ beta)))
        erro_ponderado = pesos * (p - y)
        grad = (Xb * erro_ponderado[:, None]).sum(axis=0) / n
        grad[1:] += l2 * beta[1:] / n  # L2 so nos pesos, nao no intercepto
        beta -= lr * grad
    return beta


def prever_proba(X, beta):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    return 1 / (1 + np.exp(-(Xb @ beta)))


beta = treinar_logistica(X_treino, y_treino, pesos_treino)

print("Coeficientes (intercepto + ", FEATURES, "):")
print(beta.round(4))

Coeficientes (intercepto +  ['media_causal', 'variabilidade_causal', 'tendencia_causal', 'isolamento_causal', 'cluster_baixa_recuperacao', 'cluster_queda_instavel'] ):
[-0.1617  0.3909  0.4022  0.5376 -0.0648  0.1738 -0.3348]


**Leitura dos coeficientes:** sinal positivo = aumenta a probabilidade de
queda atipica no ano seguinte; sinal negativo = reduz. Esperado:
`variabilidade_causal` e `isolamento_causal` com coeficiente positivo
(municipio mais instavel/atipico historicamente -> mais risco); o cluster
"queda_instavel" tende a coeficiente mais alto que "baixa_recuperacao" (ja
esta numa trajetoria de queda).

In [11]:
nomes_coef = ["intercepto"] + FEATURES
for nome, valor in zip(nomes_coef, beta):
    sinal = "aumenta risco" if valor > 0 else "reduz risco"
    print(f"  {nome:30s} {valor:+.4f}  ({sinal})")

  intercepto                     -0.1617  (reduz risco)
  media_causal                   +0.3909  (aumenta risco)
  variabilidade_causal           +0.4022  (aumenta risco)
  tendencia_causal               +0.5376  (aumenta risco)
  isolamento_causal              -0.0648  (reduz risco)
  cluster_baixa_recuperacao      +0.1738  (aumenta risco)
  cluster_queda_instavel         -0.3348  (reduz risco)


## Etapa E - avaliacao: AUC-ROC e AUC-PR

Como nao temos scikit-learn nesta maquina, as duas metricas sao
implementadas em numpy/pandas:
- **AUC-ROC** via formula de ranks (equivalente ao U de Mann-Whitney):
  mede se o modelo da probabilidade mais alta pros positivos do que pros
  negativos, em media -- 0.5 = aleatorio, 1.0 = perfeito.
- **AUC-PR** via area sob a curva precisao x recall (regra do trapezio) --
  mais informativa que ROC quando a classe positiva e rara, porque reflete
  diretamente "dos alertas que eu disparo, quantos sao queda atipica de
  verdade" (precisao) a cada nivel de cobertura (recall).

In [12]:
def auc_roc(y, p):
    ranks = pd.Series(p).rank(method="average").values
    n_pos, n_neg = y.sum(), len(y) - y.sum()
    soma_ranks_pos = ranks[y == 1].sum()
    return (soma_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auc_pr(y, p):
    ordem = np.argsort(-p)
    y_ord = y[ordem]
    tp = np.cumsum(y_ord)
    fp = np.cumsum(1 - y_ord)
    precisao = tp / (tp + fp)
    recall = tp / y.sum()
    return np.trapezoid(precisao, recall)


p_treino = prever_proba(X_treino, beta)
p_teste = prever_proba(X_teste, beta)

print(f"AUC-ROC treino: {auc_roc(y_treino, p_treino):.4f}")
print(f"AUC-ROC teste:  {auc_roc(y_teste, p_teste):.4f}  (0.5 = aleatorio)")
print()
print(f"AUC-PR treino: {auc_pr(y_treino, p_treino):.4f}")
print(f"AUC-PR teste:  {auc_pr(y_teste, p_teste):.4f}  (baseline aleatorio = taxa de positivos = {y_teste.mean():.4f})")

AUC-ROC treino: 0.6742
AUC-ROC teste:  0.6675  (0.5 = aleatorio)

AUC-PR treino: 0.0547
AUC-PR teste:  0.0443  (baseline aleatorio = taxa de positivos = 0.0238)


**Leitura honesta do resultado:** AUC-ROC ~0.67 no teste (bem acima de
0.5, mas longe de 1.0) e AUC-PR ~0.044 (cerca de 1.8x o baseline de 2.4%) sao
esperados para esse tipo de problema -- queda atipica de cobertura vacinal
num municipio e um evento raro cuja causa real (decisao administrativa local,
falta de insumo, campanha malsucedida, etc) nao esta capturada nas 6
features disponiveis aqui (so historico agregado de cobertura e posicao no
espaco de similaridade). O modelo tem sinal real (nao e ruido), mas serve
como ferramenta de **priorizacao/triagem** -- "olhar primeiro aqui" -- nao
como previsao precisa. Isso fica documentado explicitamente no artefato
salvo, pra nao ser mal-interpretado no frontend.

## Etapa F - faixas de risco (baixo/medio/alto) e aplicacao em 2023

**Decisao de design:** em vez de um threshold fixo arbitrario de
probabilidade, as faixas sao definidas por **percentil da distribuicao de
probabilidade prevista no conjunto de TESTE** -- mesma logica de
`percentil_isolamento` do Dia 5 (comparar com a distribuicao real observada,
nao um numero magico). Corte "alto" = top 10% de risco no teste; "medio" =
top 10-30%; "baixo" = resto. Os valores de probabilidade desses cortes
ficam fixos no artefato (nao recalculados a cada deploy).

**Aplicacao em 2023 -> risco para 2024:** aqui esta o motivo de todo o
trabalho da Etapa B ter valor pratico, nao so metodologico -- "causal no
ano T=2023" significa usar TODO o historico 2014-2023, que e exatamente o
que `clusters_municipios_mvp.csv` ja tem (media/variabilidade/tendencia/
isolamento/cluster calculados sobre 2014-2023 inteiro). Ou seja, pra
PREVER o risco de 2024 (que ainda nao tem rotulo), as features causais em
T=2023 SAO as features que o pipeline original do Dia 4/5 ja calculou --
nao e preciso recalcular nada na hora de aplicar o modelo, so usar o CSV
que ja existe. Por isso o backend (Parte 2 deste trabalho) nao precisa
rodar KMeans/k-NN -- so aplica os coeficientes salvos aqui sobre os dados
que ja estao em memoria.

In [13]:
CORTE_ALTO = float(np.percentile(p_teste, 90))   # top 10% no teste
CORTE_MEDIO = float(np.percentile(p_teste, 70))  # top 10-30% no teste

print(f"Corte 'alto risco'  (top 10% no teste): prob >= {CORTE_ALTO:.4f}")
print(f"Corte 'medio risco' (top 10-30% no teste): prob >= {CORTE_MEDIO:.4f}")

# aplica o modelo no snapshot 2014-2023 inteiro (= features causais em T=2023)
clusters_full = clusters_originais.copy()
clusters_full["cluster_baixa_recuperacao"] = clusters_full["nome_cluster"].str.contains("recuperacao").astype(float)
clusters_full["cluster_queda_instavel"] = clusters_full["nome_cluster"].str.contains("instavel").astype(float)

cont_2023 = clusters_full[
    ["media_cobertura", "variabilidade_cobertura", "tendencia_cobertura", "isolamento_similaridade"]
].values
cont_2023_padr = (cont_2023 - media_treino) / desvio_treino
X_2023 = np.hstack([cont_2023_padr, clusters_full[["cluster_baixa_recuperacao", "cluster_queda_instavel"]].values])

clusters_full["risco_2024"] = prever_proba(X_2023, beta)
clusters_full["risco_tier"] = np.where(
    clusters_full["risco_2024"] >= CORTE_ALTO, "alto",
    np.where(clusters_full["risco_2024"] >= CORTE_MEDIO, "medio", "baixo"),
)

print("\nDistribuicao de risco previsto para 2024 (5.570 municipios):")
print(clusters_full["risco_2024"].describe())
print("\nContagem por faixa:")
print(clusters_full["risco_tier"].value_counts())

Corte 'alto risco'  (top 10% no teste): prob >= 0.5756
Corte 'medio risco' (top 10-30% no teste): prob >= 0.5153

Distribuicao de risco previsto para 2024 (5.570 municipios):
count    5570.000000
mean        0.505180
std         0.093813
min         0.116692
25%         0.456764
50%         0.513658
75%         0.560263
max         0.806434
Name: risco_2024, dtype: float64

Contagem por faixa:
risco_tier
baixo    2837
medio    1682
alto     1051
Name: count, dtype: int64


In [14]:
print("Top 15 municipios de maior risco previsto para 2024:\n")
print(
    clusters_full.sort_values("risco_2024", ascending=False)
    .head(15)[["ibge6", "nome_municipio", "uf", "nome_cluster", "risco_2024"]]
    .to_string(index=False)
)

Top 15 municipios de maior risco previsto para 2024:

 ibge6                  nome_municipio uf                    nome_cluster  risco_2024
220965 São Francisco de Assis do Piauí PI Cobertura baixa, em recuperacao    0.806434
291660                      Itapitanga BA Cobertura baixa, em recuperacao    0.803150
291077                   Feira da Mata BA Cobertura baixa, em recuperacao    0.802027
430050                        Alpestre RS Cobertura baixa, em recuperacao    0.794824
430995                      Ibirapuitã RS Cobertura baixa, em recuperacao    0.789374
353310               Nova Guataporanga SP Cobertura baixa, em recuperacao    0.787406
293040                     Serra Preta BA Cobertura baixa, em recuperacao    0.787220
313130                        Ipatinga MG Cobertura baixa, em recuperacao    0.785567
291915                           Lapão BA Cobertura baixa, em recuperacao    0.781977
291150                         Gongogi BA Cobertura baixa, em recuperacao    0.781444


## Etapa G - salvar o artefato de producao

O backend (FastAPI) precisa carregar isso **uma vez no startup** (mesmo
padrao ja usado pra comunidades do grafo, ver `backend/main.py`) e so
calcular `predict()` (um produto escalar + sigmoide, muito barato) por
request -- nunca re-treinar. O artefato guarda tudo que o `predict()`
precisa: ordem das features, media/desvio de padronizacao (do treino), os
coeficientes, e os cortes de faixa de risco.

In [15]:
artefato = {
    "metadados": {
        "descricao": "Regressao logistica (numpy puro) para risco de queda atipica no ano seguinte.",
        "anos_treino": TREINO_T,
        "anos_teste": TESTE_T,
        "n_treino": int(len(treino)),
        "n_teste": int(len(teste)),
        "auc_roc_teste": float(auc_roc(y_teste, p_teste)),
        "auc_pr_teste": float(auc_pr(y_teste, p_teste)),
        "taxa_positivos_teste": float(y_teste.mean()),
        "aviso": (
            "Ferramenta de priorizacao/triagem, nao previsao precisa -- "
            "AUC-ROC ~0.67 indica sinal real mas modesto."
        ),
    },
    "features_continuas": FEATURES_CONTINUAS,
    "features_dummies": FEATURES_DUMMIES,
    "scaler_mean": media_treino.tolist(),
    "scaler_std": desvio_treino.tolist(),
    "coeficientes": {nome: float(valor) for nome, valor in zip(nomes_coef, beta)},
    "cortes_risco": {"alto": CORTE_ALTO, "medio": CORTE_MEDIO},
}

with open("../data/processed/modelo_risco.json", "w", encoding="utf-8") as f:
    json.dump(artefato, f, ensure_ascii=False, indent=2)

print("Artefato salvo em data/processed/modelo_risco.json")
print(json.dumps(artefato["metadados"], ensure_ascii=False, indent=2))

Artefato salvo em data/processed/modelo_risco.json
{
  "descricao": "Regressao logistica (numpy puro) para risco de queda atipica no ano seguinte.",
  "anos_treino": [
    2016,
    2017,
    2018,
    2019,
    2020
  ],
  "anos_teste": [
    2021,
    2022
  ],
  "n_treino": 27836,
  "n_teste": 11138,
  "auc_roc_teste": 0.667477167781019,
  "auc_pr_teste": 0.044300078662293824,
  "taxa_positivos_teste": 0.02379242233794218,
  "aviso": "Ferramenta de priorizacao/triagem, nao previsao precisa -- AUC-ROC ~0.67 indica sinal real mas modesto."
}
